# Using the clusters

In [1]:
import os
import gsw
import dask
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cf
import sys, pandas as pd
from pathlib import Path
from xhistogram.xarray import histogram as xhist

# month labels, shared by every 12-panel monthly plot below
labs = ['JAN', 'FEB', 'MAR', 'APR', 'MAY', 'JUN',
        'JUL', 'AUG', 'SEP', 'OCT', 'NOV', 'DEC']

# which cluster to process. In batch (sbatch) this is set via the CLUSTER env
# var; interactively it falls back to "F". Valid: A B1 B2 C D E F
cluster = os.environ.get("CLUSTER", "E")
print("processing cluster:", cluster)

processing cluster: B1


In [2]:
sys.path.insert(0, "/work/bk1450/b383184/Amazon/Mercator/notebooks/Analysis/"
                   "kmeans_analysis/kmean_analysis_30_180_stdZ_w")
import config as C

In [3]:
def select_cluster(group):
    """Every particle in one k-means group as a table (trajectory_id ->
    store_index + local). Reads only the label parquet, no trajectory data."""
    gid = {v: k for k, v in C.GROUP_NAMES.items()}[group]        # 'F' -> 6

    lab = pd.read_parquet(C.LABELED_FILE, columns=["trajectory_id", "cluster_group"])
    tid = lab.loc[lab.cluster_group == gid, "trajectory_id"].to_numpy()

    sel = pd.DataFrame({"trajectory_id": tid,
                        "store_index":   tid // C.TRAJ_PER_STORE,
                        "local":         tid %  C.TRAJ_PER_STORE})

    print(f"group {group} (id {gid}): {len(sel):,} particles "
          f"across {sel.store_index.nunique()} stores")
    return sel


In [4]:
sel = select_cluster(cluster) 
sel

group B1 (id 1): 2,233,199 particles across 1615 stores


,trajectory_id,store_index,local
0,9,0,9
1,49,0,49
2,51,0,51
3,64,0,64
4,94,0,94
...,...,...,...
2233194,16279954,1627,9954
2233195,16279961,1627,9961
2233196,16279963,1627,9963
2233197,16279964,1627,9964


In [5]:
from dask.distributed import Client

# Sized from the environment so that several of these notebooks can share one node
# (see run_cluster_diag_parallel.sh). The defaults reproduce the old interactive setup.
N_WORKERS = int(os.environ.get("DASK_WORKERS", 4))
N_THREADS = int(os.environ.get("DASK_THREADS", 3))
MEM_GB    = float(os.environ.get("DASK_MEM_GB", 15))

client = Client(n_workers=N_WORKERS, threads_per_worker=N_THREADS,
                memory_limit=f"{MEM_GB}GB")
print(f"dask: {N_WORKERS} workers x {N_THREADS} threads x {MEM_GB} GB "
      f"= {N_WORKERS * MEM_GB:.0f} GB budget")
client

/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 46823 instead
  warnings.warn(


dask: 3 workers x 4 threads x 12.0 GB = 36 GB budget


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:46823/status,
Dashboard: http://127.0.0.1:46823/status,Workers: 3
Total threads: 12,Total memory: 33.53 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:35411,Workers: 0
Dashboard: http://127.0.0.1:46823/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:38261,Total threads: 4
Dashboard: http://127.0.0.1:45045/status,Memory: 11.18 GiB
Nanny: tcp://127.0.0.1:35879,


In [6]:
!echo dask dashboard :D
!echo https://jupyterhub.dkrz.de/user/$USER/levante-spawner-preset/proxy/8787/status

dask dashboard :D


https://jupyterhub.dkrz.de/user/b383184/levante-spawner-preset/proxy/8787/status


In [7]:
stores1 = sorted(Path("/work/bk1450/b383184/Amazon/Mercator/data/tracks_45678").glob("Parcels_run_*_*.zarr"))
stores2  = sorted(Path("/work/bk1450/b383184/Amazon/Mercator/data/tracks_67891").glob("Parcels_run_*_*.zarr"))
stores3 = sorted(Path("/work/bk1450/b383184/Amazon/Mercator/data/tracks_78876").glob("Parcels_run_*_*.zarr"))

stores = stores1+stores2+stores3
stores[:3]

[PosixPath('/work/bk1450/b383184/Amazon/Mercator/data/tracks_45678/Parcels_run_45678_1993-01-06 00:00:00.zarr'),
 PosixPath('/work/bk1450/b383184/Amazon/Mercator/data/tracks_45678/Parcels_run_45678_1993-01-11 00:00:00.zarr'),
 PosixPath('/work/bk1450/b383184/Amazon/Mercator/data/tracks_45678/Parcels_run_45678_1993-01-16 00:00:00.zarr')]

In [8]:
# the .zarr stores that contain this cluster's particles (full paths)
cluster_stores = [stores[i] for i in sorted(sel.store_index.unique())]
print(f"cluster {cluster} lives in {len(cluster_stores)} stores")
cluster_stores[:3]


cluster B1 lives in 1615 stores


[PosixPath('/work/bk1450/b383184/Amazon/Mercator/data/tracks_45678/Parcels_run_45678_1993-01-06 00:00:00.zarr'),
 PosixPath('/work/bk1450/b383184/Amazon/Mercator/data/tracks_45678/Parcels_run_45678_1993-01-11 00:00:00.zarr'),
 PosixPath('/work/bk1450/b383184/Amazon/Mercator/data/tracks_45678/Parcels_run_45678_1993-01-16 00:00:00.zarr')]

In [9]:
ds_list = []
for si, g in sel.groupby("store_index"):
    d = xr.open_zarr(stores[si]).isel(trajectory=g.local.to_numpy())
    ds_list.append(d)
ds = xr.concat(ds_list, dim='trajectory')
ds

<xarray.Dataset> Size: 83GB
Dimensions:     (trajectory: 2233199, obs: 925)
Coordinates:
  * obs         (obs) int32 4kB 0 1 2 3 4 5 6 7 ... 918 919 920 921 922 923 924
  * trajectory  (trajectory) int64 18MB 9 49 51 64 94 ... 9961 9963 9964 9972
Data variables:
    lat         (trajectory, obs) float64 17GB dask.array<chunksize=(603, 185), meta=np.ndarray>
    lon         (trajectory, obs) float64 17GB dask.array<chunksize=(603, 185), meta=np.ndarray>
    sal         (trajectory, obs) float32 8GB dask.array<chunksize=(603, 185), meta=np.ndarray>
    temp        (trajectory, obs) float32 8GB dask.array<chunksize=(603, 185), meta=np.ndarray>
    time        (trajectory, obs) datetime64[ns] 17GB dask.array<chunksize=(603, 185), meta=np.ndarray>
    z           (trajectory, obs) float64 17GB dask.array<chunksize=(603, 185), meta=np.ndarray>
Attributes:
    Conventions:            CF-1.6/CF-1.7
    feature_type:           trajectory
    ncei_template_version:  NCEI_NetCDF_Trajectory_Template_v2.0
    parcels_kernels:        SampleParticleSampleTSAdvectionRK4_3DCheckError
    parcels_mesh:           spherical
    parcels_version:        3.1.2

In [10]:
# The stores come chunked at (19, 185) -> thousands of tiny chunks per variable, which
# is what makes the task graphs huge. Rechunk into a few big chunks first.
ds = ds.chunk({"trajectory": 5000, "obs": -1})

# One pass for the obs trim AND the histogram bounds (these used to be 5 separate reads).
num_valid_obs_steps, lat_min, lat_max, lon_min, lon_max = dask.compute(
    ds.lat.notnull().any("trajectory").sum(),
    ds.lat.min(), ds.lat.max(), ds.lon.min(), ds.lon.max(),
)
num_valid_obs_steps = int(num_valid_obs_steps)
lat_min, lat_max = float(lat_min), float(lat_max)
lon_min, lon_max = float(lon_min), float(lon_max)

ds = ds.isel(obs=slice(None, num_valid_obs_steps))

# Persist ONLY if the cluster comfortably fits in the worker memory budget. The groups
# differ by a factor of ~50: F is ~7 GB but E is 8.0M particles (~380 GB with the
# derived fields) and cannot be held in memory on a 250 GB node. Everything downstream
# is a single dask.compute, so a cluster that does not fit simply streams from disk --
# it costs one extra read, not a crash.
GB_PER_PARTICLE_OBS = (8 * 4 + 4 * 2 + 8 * 3) / 1e9   # lat/lon/z/time f8, temp/sal f4, +3 derived f8
est_gb    = ds.sizes["trajectory"] * ds.sizes["obs"] * GB_PER_PARTICLE_OBS
budget_gb = N_WORKERS * MEM_GB * 0.5                   # leave half the budget for the compute itself

if est_gb < budget_gb:
    ds = ds.persist()
    print(f"persisted: ~{est_gb:.0f} GB (budget {budget_gb:.0f} GB)")
else:
    print(f"NOT persisted: ~{est_gb:.0f} GB exceeds budget {budget_gb:.0f} GB -> streaming from disk")

ds

/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/distributed/client.py:3363: UserWarning: Sending large graph of size 57.17 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


NOT persisted: ~106 GB exceeds budget 18 GB -> streaming from disk


<xarray.Dataset> Size: 66GB
Dimensions:     (trajectory: 2233199, obs: 740)
Coordinates:
  * obs         (obs) int32 3kB 0 1 2 3 4 5 6 7 ... 733 734 735 736 737 738 739
  * trajectory  (trajectory) int64 18MB 9 49 51 64 94 ... 9961 9963 9964 9972
Data variables:
    lat         (trajectory, obs) float64 13GB dask.array<chunksize=(5000, 740), meta=np.ndarray>
    lon         (trajectory, obs) float64 13GB dask.array<chunksize=(5000, 740), meta=np.ndarray>
    sal         (trajectory, obs) float32 7GB dask.array<chunksize=(5000, 740), meta=np.ndarray>
    temp        (trajectory, obs) float32 7GB dask.array<chunksize=(5000, 740), meta=np.ndarray>
    time        (trajectory, obs) datetime64[ns] 13GB dask.array<chunksize=(5000, 740), meta=np.ndarray>
    z           (trajectory, obs) float64 13GB dask.array<chunksize=(5000, 740), meta=np.ndarray>
Attributes:
    Conventions:            CF-1.6/CF-1.7
    feature_type:           trajectory
    ncei_template_version:  NCEI_NetCDF_Trajectory_Template_v2.0
    parcels_kernels:        SampleParticleSampleTSAdvectionRK4_3DCheckError
    parcels_mesh:           spherical
    parcels_version:        3.1.2

In [11]:
ds = ds.assign(start_time=ds.time.isel(obs=0).compute())

/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/distributed/client.py:3363: UserWarning: Sending large graph of size 30.54 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/xarray/coding/times.py:650: RuntimeWarning: invalid value encountered in cast
  flat_num = flat_num.astype(np.int64)


In [12]:
lat_bins = np.linspace(lat_min, lat_max, 30)
lon_bins = np.linspace(lon_min, lon_max, 80)

## Binning

Everything below the maps is one pass: derive the extra fields (sigma0, buoyancy,
age), then bin lon/lat/release-month once for the visit count and for the weighted
sum of each variable. `dask.compute` on all of them together means the persisted
trajectories are traversed a single time, and the results land in one zarr store
(`output/means_month_{cluster}.zarr`) that the plots read back.

In [13]:
# Derived fields (density, buoyancy) computed once, up front, so the single
# binning pass below can treat them exactly like z / temp / sal.
p   = gsw.p_from_z(-ds.z, ds.lat)                  # real pressure at particle depth
SA  = gsw.SA_from_SP(ds.sal, p, ds.lon, ds.lat)    # practical -> absolute salinity
CT  = gsw.CT_from_pt(SA, ds.temp)                  # potential -> conservative temp
rho = gsw.rho(SA, CT, p)                           # IN-SITU density (real p)

g, rho_0 = 9.81, 1025
b      = (-g * (rho - rho_0) / rho_0).rename("b")   # >0 = light plume water
sigma0 = gsw.sigma0(SA, CT).rename("sigma0")        # potential density anomaly

In [14]:
ds_m = ds.assign(
    start_month=ds.start_time.dt.month,
    start_year=ds.start_time.dt.year,
    age_days=(ds.time - ds.start_time) / np.timedelta64(1, "D"),
    b=b,
    sigma0=sigma0,
)
ds_m

<xarray.Dataset> Size: 106GB
Dimensions:      (trajectory: 2233199, obs: 740)
Coordinates:
  * obs          (obs) int32 3kB 0 1 2 3 4 5 6 7 ... 733 734 735 736 737 738 739
  * trajectory   (trajectory) int64 18MB 9 49 51 64 94 ... 9961 9963 9964 9972
Data variables:
    lat          (trajectory, obs) float64 13GB dask.array<chunksize=(5000, 740), meta=np.ndarray>
    lon          (trajectory, obs) float64 13GB dask.array<chunksize=(5000, 740), meta=np.ndarray>
    sal          (trajectory, obs) float32 7GB dask.array<chunksize=(5000, 740), meta=np.ndarray>
    temp         (trajectory, obs) float32 7GB dask.array<chunksize=(5000, 740), meta=np.ndarray>
    time         (trajectory, obs) datetime64[ns] 13GB dask.array<chunksize=(5000, 740), meta=np.ndarray>
    z            (trajectory, obs) float64 13GB dask.array<chunksize=(5000, 740), meta=np.ndarray>
    start_time   (trajectory) datetime64[ns] 18MB 1993-01-06 ... 2013-08-20
    start_month  (trajectory) int64 18MB 1 1 1 1 1 1 1 1 1 ... 8 8 8 8 8 8 8 8 8
    start_year   (trajectory) int64 18MB 1993 1993 1993 1993 ... 2013 2013 2013
    age_days     (trajectory, obs) float64 13GB dask.array<chunksize=(5000, 740), meta=np.ndarray>
    b            (trajectory, obs) float64 13GB dask.array<chunksize=(5000, 740), meta=np.ndarray>
    sigma0       (trajectory, obs) float64 13GB dask.array<chunksize=(5000, 740), meta=np.ndarray>
Attributes:
    Conventions:            CF-1.6/CF-1.7
    feature_type:           trajectory
    ncei_template_version:  NCEI_NetCDF_Trajectory_Template_v2.0
    parcels_kernels:        SampleParticleSampleTSAdvectionRK4_3DCheckError
    parcels_mesh:           spherical
    parcels_version:        3.1.2

In [15]:
VARS = ["z", "temp", "sal", "sigma0", "b", "age_days"]

# Release month becomes a third histogram axis instead of a groupby, so the data is
# binned once rather than 12 times per variable.
d = ds_m.isel(obs=slice(1, None))           # skip the shared release position
month_bins = np.arange(0.5, 13.5)           # one bin per calendar month -> centres 1..12

# Chunk start_month BEFORE broadcasting. Broadcasting the in-memory (numpy) version
# first materialises a full (trajectory, obs) numpy array, which dask then ships inside
# the task graph once per histogram -- that is the "large graph" warning. Chunking the
# 1-D array first keeps the broadcast lazy, so nothing big is embedded in the graph.
month = d.start_month.chunk({"trajectory": d.lon.chunksizes["trajectory"]}).broadcast_like(d.lon)

# Samples where any variable is missing are dropped from count AND sums alike, by
# NaN-ing the coordinates: out-of-range samples never enter the histogram.
valid = d.z.notnull() & d.temp.notnull() & d.sal.notnull() & d.age_days.notnull()
lon, lat = d.lon.where(valid), d.lat.where(valid)


# --- first-arrival mask -------------------------------------------------------
# Marks the samples that are the FIRST time their particle enters that lon/lat cell.
# Binning only those turns "how old are the particles that are here" (age) into "how
# long did it take to get here" (transit time): a particle that lingers in a cell, or
# loops back into it, is counted once, at the age it first arrived.
nlon, nlat = len(lon_bins) - 1, len(lat_bins) - 1

ix = xr.apply_ufunc(np.digitize, lon, kwargs=dict(bins=lon_bins),
                    dask="parallelized", output_dtypes=[np.int64]) - 1
iy = xr.apply_ufunc(np.digitize, lat, kwargs=dict(bins=lat_bins),
                    dask="parallelized", output_dtypes=[np.int64]) - 1
inside = (ix >= 0) & (ix < nlon) & (iy >= 0) & (iy < nlat)   # NaN -> digitize -> len(bins) -> False
cell = xr.where(inside, ix * nlat + iy, -1)                  # flat cell id, -1 = not in any cell

def _first_visit(c):
    """True at the first occurrence of each cell id along the (last) obs axis."""
    out = np.zeros(c.shape, dtype=bool)
    flat, fout = c.reshape(-1, c.shape[-1]), out.reshape(-1, c.shape[-1])
    for r in range(flat.shape[0]):
        row  = flat[r]
        seen = np.flatnonzero(row >= 0)
        _, idx = np.unique(row[seen], return_index=True)     # first occurrence, in obs order
        fout[r, seen[idx]] = True
    return out

first = xr.apply_ufunc(_first_visit, cell,
                       input_core_dims=[["obs"]], output_core_dims=[["obs"]],
                       dask="parallelized", output_dtypes=[bool])

lon_f, lat_f = lon.where(first), lat.where(first)            # arrival samples only


def binned(x, y, weights=None):
    return xhist(x, y, month,
                 bins=[lon_bins, lat_bins, month_bins],
                 dim=["trajectory", "obs"], weights=weights,
                 bin_dim_suffix="")

counts    = binned(lon, lat)                                 # particle-visit counts (obs summed)
sums      = {v: binned(lon, lat, d[v]) for v in VARS}        # weighted sums
n_arrived = binned(lon_f, lat_f)                             # DISTINCT particles that reach the cell
t_arrived = binned(lon_f, lat_f, d.age_days)                 # sum of their arrival ages

# ONE traversal of the persisted data for every count and sum
counts, sums, n_arrived, t_arrived = dask.compute(counts, sums, n_arrived, t_arrived)

means = xr.Dataset({v: (sums[v] / counts.where(counts > 0)).rename(v) for v in VARS})
means = means.rename(age_days="age").assign(counts=counts)

# transit time: mean age at FIRST arrival, i.e. how long the particles that get here
# took to do so. Distinct from `age`, which averages over every visit and so is
# inflated wherever particles linger or recirculate.
means["transit"]   = t_arrived / n_arrived.where(n_arrived > 0)
means["n_arrived"] = n_arrived

# fraction of the month's released particles that ever reach the cell. Transit time is
# only meaningful where this is non-negligible: a cell reached by 3 particles out of
# 12,000 has a transit time, but not one you would want to quote.
n_rel = (ds_m.start_month.to_series().value_counts()
         .reindex(range(1, 13), fill_value=0).sort_index())
n_released = xr.DataArray(n_rel.values, coords={"start_month": means.start_month},
                          dims="start_month")
means["reached"] = n_arrived / n_released

means.drop_encoding().to_zarr(f"output/means_month_{cluster}.zarr", mode="w")
means

/work/bk1450/b383184/conda/envs/parcels_3.1.2/lib/python3.13/site-packages/distributed/client.py:3363: UserWarning: Sending large graph of size 188.64 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


<xarray.Dataset> Size: 2MB
Dimensions:      (lon: 79, lat: 29, start_month: 12)
Coordinates:
  * lon          (lon) float64 632B -86.8 -86.19 -85.58 ... -40.42 -39.81 -39.2
  * lat          (lat) float64 232B -1.496 -0.4148 0.6661 ... 26.61 27.69 28.77
  * start_month  (start_month) float64 96B 1.0 2.0 3.0 4.0 ... 10.0 11.0 12.0
Data variables:
    z            (lon, lat, start_month) float64 220kB nan nan nan ... nan nan
    temp         (lon, lat, start_month) float64 220kB nan nan nan ... nan nan
    sal          (lon, lat, start_month) float64 220kB nan nan nan ... nan nan
    sigma0       (lon, lat, start_month) float64 220kB nan nan nan ... nan nan
    b            (lon, lat, start_month) float64 220kB nan nan nan ... nan nan
    age          (lon, lat, start_month) float64 220kB nan nan nan ... nan nan
    counts       (lon, lat, start_month) int64 220kB 0 0 0 0 0 0 ... 0 0 0 0 0 0
    transit      (lon, lat, start_month) float64 220kB nan nan nan ... nan nan
    n_arrived    (lon, lat, start_month) int64 220kB 0 0 0 0 0 0 ... 0 0 0 0 0 0
    reached      (lon, lat, start_month) float64 220kB 0.0 0.0 0.0 ... 0.0 0.0

In [16]:
# everything the plots need now lives in one file -- restart here without recomputing
means = xr.open_dataset(f"output/means_month_{cluster}.zarr").compute()
means

<xarray.Dataset> Size: 2MB
Dimensions:      (lon: 79, lat: 29, start_month: 12)
Coordinates:
  * lat          (lat) float64 232B -1.496 -0.4148 0.6661 ... 26.61 27.69 28.77
  * lon          (lon) float64 632B -86.8 -86.19 -85.58 ... -40.42 -39.81 -39.2
  * start_month  (start_month) float64 96B 1.0 2.0 3.0 4.0 ... 10.0 11.0 12.0
Data variables:
    age          (lon, lat, start_month) float64 220kB nan nan nan ... nan nan
    b            (lon, lat, start_month) float64 220kB nan nan nan ... nan nan
    counts       (lon, lat, start_month) int64 220kB 0 0 0 0 0 0 ... 0 0 0 0 0 0
    n_arrived    (lon, lat, start_month) int64 220kB 0 0 0 0 0 0 ... 0 0 0 0 0 0
    reached      (lon, lat, start_month) float64 220kB 0.0 0.0 0.0 ... 0.0 0.0
    sal          (lon, lat, start_month) float64 220kB nan nan nan ... nan nan
    sigma0       (lon, lat, start_month) float64 220kB nan nan nan ... nan nan
    temp         (lon, lat, start_month) float64 220kB nan nan nan ... nan nan
    transit      (lon, lat, start_month) float64 220kB nan nan nan ... nan nan
    z            (lon, lat, start_month) float64 220kB nan nan nan ... nan nan

In [17]:
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from matplotlib.colors import BoundaryNorm
import cmocean as cm

proj = ccrs.PlateCarree()

# --- per-variable plotting config: colormap, colorbar label, discrete levels ---
# add a new variable by adding one entry to each of the three dicts.
cmaps = {
    "counts":  "Spectral_r",
    "z":       "turbo",
    "temp":    cm.cm.balance,
    "sal":     "jet",
    "sigma0":  'Spectral',          # sequential: darker = denser
    "b":       cm.cm.balance,    # diverging around 0: buoyant vs. dense
    "age":     "tab10",
    "transit": 'tab10',   # 7 bands = the 7 intervals in LEVELS["transit"]
    "reached": "cividis",
}
labels = {
    "counts":  "log10(counts)",
    "z":       "mean depth [m]",
    "temp":    "mean temperature [°C]",
    "sal":     "mean salinity",
    "sigma0":  r"mean $\sigma_0$ [kg m$^{-3}$]",
    "b":       r"mean buoyancy [m s$^{-2}$]",
    "age":     "mean particle age [days]",
    "transit": "mean transit time to first arrival [days]",
    "reached": "fraction of released particles reaching the cell",
}
LEVELS = {
    "counts":  None,             # None -> let contourf pick the levels
    "z":       [0, 50, 100, 150, 200, 300, 400, 500],
    "temp":    [0, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30],
    "sal":     np.insert(np.arange(30, 36, 0.5), 0, 0),
    "sigma0":  [5, 10, 15, 18, 20, 21, 22, 23, 24, 25, 26, 27],   # <15 (plume) clipped to lowest band
    "b":       np.arange(-0.02, 0.16, 0.01),                      # >0 = light plume water
    "age":     np.arange(0, 180, 30),   # fixed across clusters so A-F are comparable
    # fine near the source (fast arrivals resolve at 10 d), coarser far afield;
    # >180 d saturates the top band. Fixed, so clusters A-F compare directly.
    "transit": [0, 10, 30, 60, 90, 120, 150, 180],
    "reached": [0, 0.01, 0.02, 0.05, 0.1, 0.2, 0.4, 0.6, 0.8, 1.0],
}

# contourf draws at zorder 1, so land/coastlines must sit above it or the filled
# contours bury the continents (very visible on `reached`, which is 0 -- not NaN --
# over land and so fills the whole domain).
Z_DATA, Z_LAND, Z_COAST = 1, 3, 4


def plot_monthly(da, name):
    """12-panel monthly map of `da`. Colormap, colorbar label and discrete
    contour levels are looked up from the cmaps/labels/LEVELS dicts by `name`."""
    cmap, label, levels = cmaps[name], labels[name], LEVELS[name]
    cmap = plt.get_cmap(cmap) if isinstance(cmap, str) else cmap
    norm = BoundaryNorm(levels, ncolors=cmap.N, clip=True) if levels is not None else None

    fig, ax = plt.subplots(4, 3, figsize=(12, 8), subplot_kw=dict(projection=proj))
    ax = ax.ravel()

    for i in range(12):
        dd = da.sel(start_month=i + 1)

        pcm = dd.plot.contourf(
            x="lon", y="lat", ax=ax[i], transform=proj,
            cmap=cmap, levels=levels, norm=norm, add_colorbar=False,
            zorder=Z_DATA,
        )

        ax[i].add_feature(cf.LAND, facecolor="#8e9497ff", zorder=Z_LAND)
        ax[i].coastlines(zorder=Z_COAST)
        ax[i].set_title(labs[i], fontsize=10)
        ax[i].set_xticks(
            np.arange(np.floor(dd.lon.min()), np.ceil(dd.lon.max()) + 1e-6, 10), crs=proj
        )
        ax[i].set_yticks(
            np.arange(np.floor(dd.lat.min()), np.ceil(dd.lat.max()) + 1e-6, 5), crs=proj
        )
        ax[i].xaxis.set_major_formatter(LongitudeFormatter(number_format=".0f", degree_symbol="°"))
        ax[i].yaxis.set_major_formatter(LatitudeFormatter(number_format=".0f", degree_symbol="°"))
        ax[i].tick_params(labelsize=8)
        ax[i].set_xlabel("")
        ax[i].set_ylabel("")

    fig.subplots_adjust(
        left=0.05, right=0.95,
        bottom=0.08, top=0.90,
        wspace=0.05, hspace=0.4
    )
    cbar = fig.colorbar(pcm, ax=ax, orientation="horizontal", fraction=0.05, pad=0.10,
                        ticks=levels)
    cbar.ax.set_xlabel(label, fontsize=10)
    plt.savefig(f"figures/mean_{name}_LPT_{cluster}.png", dpi=300, bbox_inches="tight")
    plt.show()

In [18]:
log_counts = np.log10(means.counts.where(means.counts > 0))
plot_monthly(log_counts, "counts")

## Maps: depth, temperature, salinity

In [19]:
plot_monthly(means.z, "z")

In [20]:
plot_monthly(means.temp, "temp")

In [21]:
plot_monthly(means.sal, "sal")

## Maps: density and buoyancy

In [22]:
plot_monthly(means.sigma0, "sigma0")

In [23]:
plot_monthly(means.b, "b")

## Horizontal $\sigma_0$ gradient (density fronts / geostrophic shear)

`|∇σ₀|` on the monthly-mean field: large values mark density fronts (plume edge,
retroflection, NBC) and, via thermal wind, strong vertical shear of the
geostrophic flow. NB this is a Lagrangian depth-and-time-mixed mean, so read it
as *where the fronts are*, not as a quantitative geostrophic shear.

In [24]:
# horizontal gradient of monthly-mean sigma0 (metric-aware: cos(lat) on lon)
R     = 6371e3
deg2m = np.deg2rad(1) * R                       # metres per degree of latitude

s = means.sigma0
dsig_dx = s.differentiate("lon") / (deg2m * np.cos(np.deg2rad(s.lat)))
dsig_dy = s.differentiate("lat") /  deg2m
grad_sigma0 = (np.hypot(dsig_dx, dsig_dy) * 1e5).rename("grad_sigma0")  # kg m-3 / 100 km

# register in the plotting config (one entry per dict, like every other variable)
cmaps["grad_sigma0"]  = 'tab10'
labels["grad_sigma0"] = r"$|\nabla\sigma_0|$ [kg m$^{-3}$ / 100 km]"
LEVELS["grad_sigma0"] = np.arange(1.8, 4, 0.2)   # fronts (>2) saturate the top band

In [25]:
plot_monthly(grad_sigma0, "grad_sigma0")

## Particle age

In [26]:
plot_monthly(means.age, "age")

## Transit time — how long the particles take to get there

Not the same as particle age. `age` averages over *every* sample in a cell, so a cell
where particles linger for months, or that the same particle re-enters on a
recirculation, reads old even if it is reached quickly. `transit` averages, over the
particles that ever arrive, the age at which each one **first** enters the cell — the
travel time from the release point.

Read it together with `reached` (below): where only a handful of particles ever arrive,
the transit time is an average over those few and is not robust. `age >> transit` marks
retention (particles arrive early and stay); `age ≈ transit` marks flow-through.

In [27]:
plot_monthly(means.transit, "transit")

In [28]:
plot_monthly(means.reached, "reached")

In [29]:
# the defensible version: transit time only where at least 1% of the month's particles
# arrive, so the map isn't dominated by cells a couple of strays wandered into
transit_ok = means.transit.where(means.reached >= 0.01)

cmaps["transit_ok"]  = cmaps["transit"]
labels["transit_ok"] = "mean transit time [days]  (cells reached by >1% of particles)"
LEVELS["transit_ok"] = LEVELS["transit"]

plot_monthly(transit_ok, "transit_ok")